In [1]:
import pandas as pd
import os
import collections
import time
from helper_functions import (
    convert_samples_to_binary,
    get_column_data_types,
    add_id_column,
    organize_photos_in_folders
)

In [2]:
# Load the dataset
try:
    df = pd.read_csv("../../Datasets/csv/UAVATTACK/spoofing-merged-gps-only.csv")
except FileNotFoundError:
    print("Error: The file '../../Datasets/csv/UAVATTACK/spoofing-merged-gps-only.csv' was not found.")
    print("Please make sure the path is correct.")
    exit()

In [3]:
# Data Preprocessing

# Clean up column names
df.columns = df.columns.str.replace('[\\[\\]<>]', '_', regex=True)

# Drop label and timestamp columns from features
drop_cols = ['timestamp', 'label']
X = df.drop(columns=drop_cols)
y = df['label']

# Encode y if needed
if y.dtype == object or y.dtype.name == 'category':
    from sklearn.preprocessing import LabelEncoder
    le = LabelEncoder()
    y = le.fit_transform(y)

# Add an 'Id' column for image naming, using add_id_column from helper_functions
classification_id = 'label'
# Add the label column back to df if it was dropped earlier
if 'label' not in df.columns:
    df[classification_id] = y
df = add_id_column(df, classification_id)  # This will add an 'Id' column like '0-1', '1-1', etc.

# Prepare features (X) and target (y) for image conversion
X = df.drop([classification_id, 'Id', 'timestamp'], axis=1, errors='ignore')
y = df[classification_id]
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Row count match: {X.shape[0] == y.shape[0]}")


X shape: (3622, 82)
y shape: (3622,)
Row count match: True


In [4]:
# Image Generation

# Update this path to where you want images to be saved
image_directory = "image-results-spoofing"
os.makedirs(image_directory, exist_ok=True)

# Get feature types: 0 = float, 1 = int, 2 = bool (check your dataset)
feature_types = get_column_data_types(X)
print("Feature types:", feature_types)

# Precision and binary encoding settings
precision = 64     # number of binary digits per feature
one = 128          # pixel value for '1'
zero = 0           # pixel value for '0'
n_jobs = -1        # use all available cores

# Convert dataset samples to binary images
start = time.time()
convert_samples_to_binary(
    X,
    df["Id"],
    image_directory,
    precision,
    one,
    zero,
    n_jobs,
    feature_types
)
end = time.time()
print(f"Time to convert data to images: {end - start:.2f} seconds")

# Organize images into folders by class
organize_photos_in_folders(image_directory, y)

Feature types: [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0]
Time to convert 3622 samples to binary: 6.036913871765137 seconds. Seconds per sample = 0.0016667349176601703
Time to convert data to images: 6.04 seconds
3622
{'benign': 3124, 'malicious': 498}


In [5]:
# Print class distribution
print(collections.Counter(y))

Counter({'benign': 3124, 'malicious': 498})
